# ORACLYS — Hidden Pattern Discovery Engine
### An exploratory data-science notebook

**Goal:** combine multiple variables, discover candidate relationships, identify unusual observations, and create an evidence-led map of patterns worth investigating.

> This is a research prototype, not an automated causal-discovery or decision-making system. Correlation and feature importance do not establish causation. Validate findings with domain knowledge and suitable study designs.

**Notebook workflow**
1. Load a sample dataset or your own CSV.
2. Inspect data quality and prepare numeric features.
3. Discover statistical associations.
4. Find groups and unusual observations.
5. Build a relationship network.
6. Export a concise discovery report.

## 1. Install and import libraries

Run the install cell only if a required package is missing in your environment.

In [ ]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn networkx scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

print("ORACLYS environment ready.")

## 2. Load data

By default, this notebook creates a **synthetic public-health-style dataset** so every cell can run immediately. Replace `USE_SAMPLE_DATA` with `False` and set `DATA_PATH` to your CSV when you are ready.

The synthetic data is for demonstration only; it is not real patient or population evidence.

In [ ]:
USE_SAMPLE_DATA = True
DATA_PATH = "your_dataset.csv"  # e.g. "data/community_health.csv"

def make_sample_data(n=500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    distance = rng.gamma(2.2, 3.0, n)
    transport = np.clip(100 - 5.2 * distance + rng.normal(0, 13, n), 0, 100)
    income = np.clip(rng.normal(26000, 11000, n), 4000, 90000)
    facility_capacity = np.clip(rng.normal(65, 17, n), 10, 100)
    wait_days = np.clip(
        1.5 + 0.45 * distance - 0.025 * transport
        + 0.018 * (100 - facility_capacity)
        + rng.normal(0, 2.2, n), 0, None
    )
    access_score = np.clip(
        85 - 3.0 * distance + 0.12 * transport
        + 0.00035 * income + rng.normal(0, 9, n), 0, 100
    )
    df = pd.DataFrame({
        "distance_km": distance,
        "transport_availability": transport,
        "monthly_household_income": income,
        "facility_capacity": facility_capacity,
        "waiting_days": wait_days,
        "healthcare_access_score": access_score,
    })
    # Introduce a small amount of missingness for data-quality demonstration.
    for col in ["transport_availability", "monthly_household_income"]:
        mask = rng.random(n) < 0.04
        df.loc[mask, col] = np.nan
    return df

if USE_SAMPLE_DATA:
    df = make_sample_data()
else:
    df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Data quality audit

Check types, missingness, duplicates, and descriptive statistics before searching for patterns.

In [ ]:
def data_quality_report(data):
    report = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
        "unique_values": data.nunique(dropna=True),
    })
    return report.sort_values("missing_percent", ascending=False)

display(data_quality_report(df))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 4. Prepare numeric features

This first version focuses on numeric columns. Categorical variables need an explicit encoding strategy before inclusion. Highly identifier-like columns should generally be excluded.

In [ ]:
numeric_df = df.select_dtypes(include=np.number).copy()
if numeric_df.shape[1] < 2:
    raise ValueError("ORACLYS needs at least two numeric columns to discover relationships.")

# Remove columns that contain no observed values or are constant.
numeric_df = numeric_df.dropna(axis=1, how="all")
numeric_df = numeric_df.loc[:, numeric_df.nunique(dropna=True) > 1]

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(numeric_df),
    columns=numeric_df.columns,
    index=numeric_df.index
)

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_imputed),
    columns=X_imputed.columns,
    index=X_imputed.index
)

print("Numeric features retained:", list(X_imputed.columns))
display(X_imputed.head())

## 5. Association discovery

Calculate both Pearson correlation (linear association) and Spearman correlation (monotonic association). These are exploratory associations, not causal effects.

In [ ]:
pearson_corr = X_imputed.corr(method="pearson")
spearman_corr = X_imputed.corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(pearson_corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Pearson correlation")
sns.heatmap(spearman_corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Spearman correlation")
plt.tight_layout()
plt.show()

# Rank unique feature pairs by absolute Spearman association.
pairs = []
cols = list(spearman_corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = cols[i], cols[j]
        rho, p_value = spearmanr(X_imputed[a], X_imputed[b])
        pairs.append((a, b, rho, p_value, abs(rho)))

association_table = pd.DataFrame(
    pairs, columns=["feature_a", "feature_b", "spearman_rho", "p_value", "abs_rho"]
).sort_values("abs_rho", ascending=False)

display(association_table.head(15))

## 6. Non-linear dependency screening with mutual information

Mutual information can detect dependencies that correlation may miss. Scores are not signed, do not indicate direction, and should be treated as screening signals.

In [ ]:
mi_rows = []
for target in X_imputed.columns:
    predictors = [c for c in X_imputed.columns if c != target]
    mi = mutual_info_regression(
        X_imputed[predictors], X_imputed[target],
        random_state=RANDOM_STATE
    )
    mi_rows.extend(zip([target] * len(predictors), predictors, mi))

mi_table = pd.DataFrame(
    mi_rows, columns=["target", "candidate_feature", "mutual_information"]
).sort_values("mutual_information", ascending=False)

display(mi_table.head(20))

## 7. Discover candidate groups with clustering

K-means is used here as a baseline. The silhouette score is a rough measure of cluster separation; it does not prove that clusters represent meaningful real-world categories.

In [ ]:
cluster_scores = []
max_k = min(7, len(X_scaled) - 1)
for k in range(2, max_k + 1):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_scaled)
    cluster_scores.append((k, silhouette_score(X_scaled, labels)))

score_df = pd.DataFrame(cluster_scores, columns=["k", "silhouette_score"])
display(score_df)

best_k = int(score_df.sort_values("silhouette_score", ascending=False).iloc[0]["k"])
cluster_model = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = cluster_model.fit_predict(X_scaled)

clustered = df.copy()
clustered["oraclys_cluster"] = cluster_labels
display(clustered.groupby("oraclys_cluster").mean(numeric_only=True).round(2))

plt.figure(figsize=(8, 4))
sns.barplot(data=score_df, x="k", y="silhouette_score")
plt.title("Cluster separation screening")
plt.show()

## 8. Detect unusual observations

Isolation Forest flags observations that differ from the broader feature distribution. An anomaly is not automatically an error, fraud event, or high-risk case; inspect it in context.

In [ ]:
iso = IsolationForest(contamination=0.03, random_state=RANDOM_STATE)
anomaly_labels = iso.fit_predict(X_scaled)

anomaly_results = df.copy()
anomaly_results["anomaly_flag"] = np.where(anomaly_labels == -1, "Review", "Typical")
anomaly_results["anomaly_score"] = iso.decision_function(X_scaled)

print("Flagged observations:", int((anomaly_labels == -1).sum()))
display(anomaly_results[anomaly_results["anomaly_flag"] == "Review"].head(20))

## 9. Build a relationship network

Each feature is a node. An edge connects two features when the absolute Spearman association crosses the selected threshold. The graph is a visual index of candidate relationships, not proof of causal connections.

In [ ]:
EDGE_THRESHOLD = 0.35

G = nx.Graph()
G.add_nodes_from(X_imputed.columns)

for row in association_table.itertuples(index=False):
    if row.abs_rho >= EDGE_THRESHOLD:
        G.add_edge(
            row.feature_a, row.feature_b,
            weight=float(row.abs_rho),
            rho=float(row.spearman_rho)
        )

plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.9)
widths = [1 + 4 * G[u][v]["weight"] for u, v in G.edges()]
nx.draw_networkx(
    G, pos, with_labels=True, node_size=2200,
    font_size=8, width=widths, edge_color="gray"
)
edge_labels = {(u, v): f'{G[u][v]["rho"]:.2f}' for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title(f"ORACLYS relationship network | |Spearman rho| ≥ {EDGE_THRESHOLD}")
plt.axis("off")
plt.show()

print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())

## 10. Generate a discovery summary

This summary surfaces the strongest observed associations, the best silhouette score from the tested cluster counts, and the anomaly count. Interpret each result in the context of the dataset and study design.

In [ ]:
summary_lines = [
    "# ORACLYS Discovery Summary",
    "",
    f"- Rows analysed: {len(df)}",
    f"- Numeric features retained: {len(X_imputed.columns)}",
    f"- Duplicate rows: {int(df.duplicated().sum())}",
    f"- Observations flagged for review: {int((anomaly_labels == -1).sum())}",
    f"- Relationship network edges: {G.number_of_edges()}",
    "",
    "## Strongest Spearman associations"
]

for row in association_table.head(10).itertuples(index=False):
    summary_lines.append(
        f"- {row.feature_a} ↔ {row.feature_b}: rho={row.spearman_rho:.3f}, p={row.p_value:.3g}"
    )

summary_lines += [
    "",
    "## Clustering screening",
    f"- Best silhouette score among tested k values: k={best_k}, score={score_df['silhouette_score'].max():.3f}",
    "",
    "## Interpretation safeguards",
    "- Associations are not causal effects.",
    "- P-values are exploratory and are not adjusted for multiple comparisons.",
    "- Clusters and anomalies require domain review.",
    "- Synthetic sample data must never be presented as real-world findings."
]

summary_text = "\n".join(summary_lines)
print(summary_text)

with open("oraclys_discovery_summary.md", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved: oraclys_discovery_summary.md")

## 11. Export results

Export the association table, cluster assignments, and anomaly review table for further analysis.

In [ ]:
association_table.to_csv("oraclys_associations.csv", index=False)
clustered.to_csv("oraclys_clustered_data.csv", index=False)
anomaly_results.to_csv("oraclys_anomaly_review.csv", index=False)

print("Exported:")
print("- oraclys_associations.csv")
print("- oraclys_clustered_data.csv")
print("- oraclys_anomaly_review.csv")
print("- oraclys_discovery_summary.md")

## Next development milestones

- Add categorical-variable encoding and mixed-data association measures.
- Add bootstrap confidence intervals and multiple-testing correction.
- Add time-series change detection for longitudinal datasets.
- Add domain-specific validation rules and provenance tracking.
- Build a Streamlit or React front end around this analysis pipeline.
- Add tests, configuration files, logging, and reproducible experiment tracking.

**Research discipline:** ORACLYS should report what the data supports, what remains uncertain, and what evidence would be needed to investigate a candidate pattern further.